In [0]:
import sys
from pyspark.sql import functions as F, Window
 
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD)")
 
sys.path.append("/Workspace/Shared")
import aws_utils

env              = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("run_date").strip()
 
if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primaryactive.exposure")
        .filter(F.col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

TARGET_ASSET = f"`{env}-cdh-delivery`.`it`.`l4738_ref_asset`"

print(f"env={env} | run_date={run_date}")
print(f"target      : {TARGET_ASSET}")

In [0]:
from pyspark.sql import functions as F, Window

ACTIVE = ["ΕΝΕΡΓΗ", "Ενεργή", "Ενεργη", "ενεργή", "ενεργη", "ΟΥΣΙΑΣ", "Ενεργό", "N/A"]

w_coll = Window.partitionBy(
    F.coalesce(F.col("CollateralOldKey"), F.col("CollateralKey")),
    "CollateralKey", "AgreementKey", "OwnershipKey", "InitialLegalOrder"
).orderBy("CollateralId")

collateral_re = (
    spark.table(f"`{env}-cdh`.primaryactive.collateral")
    .filter(
        F.col("RealEstateKey").isNotNull()
        & (F.col("BusinessDate") == run_date)
        & F.col("CollateralStatus").isin(ACTIVE)
        & F.col("RealEstateCollateralStatus").isin(ACTIVE)
        & F.col("PortfolioName").isNotNull()
    )
    .withColumn("rn", F.row_number().over(w_coll))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

pf_lookup = (
    spark.table(f"`{env}-cdh`.primaryactive.portfolio")
    .filter(
        (F.col("BusinessDate") == run_date)
        & F.col("unit_code_profits").isNotNull()
    )
    .select(
        F.col("portfolio_id_thesis").alias("pf_thesis_id"),
        F.col("PortfolioId").alias("pf_PortfolioId"),
        F.col("TiresiasName").alias("pf_TiresiasName"),
        F.coalesce(
            F.col("vendor_code_qualco").cast("string"),
            F.col("spv_qualco_AB").cast("long").cast("string"),
            F.col("unit_code_profits").cast("long").cast("string")
        ).alias("pf_vendorCode")
    )
    .dropDuplicates(["pf_thesis_id"])
)

collateral_re = (
    collateral_re
    .join(pf_lookup, F.col("PortfolioKey").cast("int") == F.col("pf_thesis_id"), "left")
    .drop("pf_thesis_id")
)

w_reb = Window.partitionBy("RealEstateKey").orderBy(
    F.when(
        (F.col("RealEstateBuildingType") == "Real Estate") & F.col("SourceTableName").isNull(), 0
    ).otherwise(1),
    "RealEstateBuildingKey"
)

reb = (
    spark.table(f"`{env}-cdh`.primaryactive.realestatebuilding")
    .filter(F.col("BusinessDate") == run_date)
    .withColumn("rn", F.row_number().over(w_reb))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("RealEstateKey").alias("reb_key"),
        F.col("RealEstateCategory").alias("ParentCategory"),
        F.col("RealEstateSubCategory").alias("PropertyCategory"),
        F.col("RealEstateDescription").alias("PropertyDescription"),
        F.col("RealEstateMarketValue").cast("decimal(19,2)").alias("MarketValue"),
        F.coalesce(
            F.col("RealEstateValueDate").cast("date"),
            F.col("RealEstateEvaluationDate").cast("date"),
            F.lit("1900-01-01").cast("date")
        ).alias("LastValuationDate"),
        F.col("RealEstateCity").alias("city"),
        F.col("RealEstateArea").alias("area"),
        F.col("RealEstatePostalCode").alias("ZipCode"),
        F.col("RealEstateAddress1").alias("Address"),
        F.col("RealEstateDistrict").alias("Prefecture"),
        F.col("RealEstateCepalKey").alias("PropertyCepalKey"),
    )
)

w_pty = Window.partitionBy("PartyKey", "PortfolioKey").orderBy("PartyID")

pty = (
    spark.table(f"`{env}-cdh`.primaryactive.party")
    .filter(F.col("BusinessDate") == run_date)
    .withColumn("rn", F.row_number().over(w_pty))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("PartyKey").alias("pty_key"),
        F.col("PortfolioKey").alias("pty_PortfolioKey"),
        F.col("TaxRegistrationNumber").alias("OwnerTaxNumber"),
        F.trim(
            F.coalesce(
                F.col("LegalNameGR"),
                F.concat(F.col("FirstNameGR"), F.lit("  "), F.col("LastNameGR"))
            )
        ).alias("PropertyOwnerName"),
        F.col("BirthDate").cast("date").alias("BirthDate"),
    )
)

bd_source = (
    spark.table(f"`{env}-cdh`.derivativeactive.dparty")
    .filter(
        (F.col("BusinessDate") == run_date)
        & F.col("BirthDate").isNotNull()
        & F.col("TaxRegistrationNumber").isNotNull()
    )
    .select(
        F.col("TaxRegistrationNumber").alias("bd_AFM"),
        F.col("BirthDate").cast("date").alias("bd_from_dparty"),
    )
    .dropDuplicates(["bd_AFM"])
)

pty = (
    pty
    .join(bd_source, pty["OwnerTaxNumber"] == bd_source["bd_AFM"], "left")
    .withColumn("BirthDate", F.coalesce(F.col("BirthDate"), F.col("bd_from_dparty")))
    .drop("bd_AFM", "bd_from_dparty")
)

df_asset = (
    collateral_re
    .join(reb,
          collateral_re["RealEstateKey"].cast("string") == reb["reb_key"],
          "inner")
    .join(pty,
          (collateral_re["OwnerPartyKey"].cast("string") == pty["pty_key"])
          & (collateral_re["PortfolioKey"] == pty["pty_PortfolioKey"]),
          "left")
    .filter(F.col("InitialLegalOrder").isNotNull())  
    .select(
        collateral_re["BusinessDate"].cast("date").alias("datadate"),
        F.col("pf_PortfolioId").cast("int").alias("PortfolioId"),
        F.col("pf_TiresiasName").alias("Portfolio"),
        F.col("pf_vendorCode").alias("vendorCode"),
        F.coalesce(
            collateral_re["CollateralOldKey"],
            F.concat(F.lit("N/A - "), collateral_re["CollateralKey"])
        ).alias("OldCollateralId"),
        collateral_re["SubCategory"].alias("CollateralType"),
        collateral_re["CollateralStatus"],
        collateral_re["CollateralAmount"].cast("decimal(19,2)"),
        collateral_re["AgreementKey"].alias("AgreementNumber"),
        reb["PropertyCepalKey"].alias("PropertyKey"),
        collateral_re["InitialLegalOrder"].cast("int").alias("lien"),
        reb["ParentCategory"],
        reb["PropertyCategory"],
        reb["PropertyDescription"],
        reb["MarketValue"],
        reb["LastValuationDate"],
        reb["city"],
        reb["area"],
        reb["ZipCode"],
        reb["Address"],
        reb["Prefecture"],
        collateral_re["OwnershipKey"].cast("int").alias("PropertyOwnershipId"),
        pty["OwnerTaxNumber"],
        pty["PropertyOwnerName"],
        collateral_re["OwnershipPercentage"].cast("decimal(19,6)").alias("OwnershipPercentage"),
        collateral_re["RealTitleDescription"].alias("OwnershipType"),
        pty["BirthDate"],
        F.current_date().alias("EtlLoadingDate"),
    )
)

row_count = df_asset.count()
print(f"ref_asset records: {row_count:,}  (run_date={run_date})")
assert row_count > 0, f"0 rows — check run_date={run_date} and active status filters"
display(df_asset.limit(5))

In [0]:
from pyspark.sql import functions as F, Window

TARGET_NONRE = f"`{env}-cdh-delivery`.`it`.`l4738_ref_nonre`"

coll_nre = (
    spark.table(f"`{env}-cdh`.primaryactive.collateral")
    .filter(
        (F.col("BusinessDate") == run_date)
        & F.col("NonRealEstateKey").isNotNull()
        & F.col("CollateralStatus").isin(ACTIVE)
        & F.col("NonRealEstateCollateralStatus").isin(ACTIVE)
        & F.col("PortfolioName").isNotNull()
    )
    .join(pf_lookup, F.col("PortfolioKey").cast("int") == F.col("pf_thesis_id"), "left")
    .drop("pf_thesis_id")
)

nre = (
    spark.table(f"`{env}-cdh`.primaryactive.nonrealestate")
    .filter(F.col("BusinessDate") == run_date)
    .select(
        F.col("NonRealEstateKey").alias("nre_key"),
        F.trim(F.col("NonRealEstateItemDescription")).alias("detItemDescr"),
        F.col("ISIN"),
        F.trim(F.col("NonRealEstateDescription")).alias("det_descr"),
    )
)

w_nre = Window.partitionBy(
    "BusinessDate", "PortfolioKey",
    "CollateralOldKey", "CollateralCode",
    "AgreementKey", "CollateralCurrency", "CollateralStatusChangeDate",
    "CollateralAmount", "ISIN",
    "Category", "SubCategory"
).orderBy(
    F.col("detItemDescr").isNull().asc(),
    F.desc(F.coalesce(F.col("detItemDescr"), F.lit(""))),
    F.desc(F.coalesce(F.col("det_descr"),    F.lit("")))
)

df_nonre = (
    coll_nre
    .join(nre, F.col("NonRealEstateKey").cast("string") == F.col("nre_key"), "inner")
    .withColumn("rn", F.row_number().over(w_nre))
    .filter(F.col("rn") == 1)
    .drop("rn", "nre_key")
    .select(
        F.col("BusinessDate").cast("date").alias("datadate"),
        F.col("pf_PortfolioId").cast("int").alias("PortfolioId"),
        F.col("pf_TiresiasName").alias("Portfolio"),
        F.col("pf_vendorCode").alias("vendorCode"),
        F.coalesce(F.col("CollateralOldKey"), F.concat(F.lit("N/A - "), F.col("CollateralCode"))).alias("OldCollateralId"),
        F.col("AgreementKey").alias("AgreementNumber"),
        F.col("CollateralCurrency").alias("Currency"),
        F.col("CollateralStatusChangeDate"),
        F.col("CollateralAmount").cast("decimal(19,2)").alias("collateralAmount"),
        F.col("CollateralAmount").cast("decimal(19,2)").alias("Det_collateralAmount"),
        F.col("detItemDescr"),
        F.col("ISIN").alias("isin"),
        F.lit("1").alias("lien_type"),
        F.lit("1900-01-01").cast("date").alias("latestEvaluationDate"),
        F.col("det_descr"),
        F.col("Category"),
        F.col("SubCategory"),
        F.col("CollateralCode").alias("thesisCollaterId"),
        F.current_date().alias("EtlLoadingDate"),
    )
)

row_count_nonre = df_nonre.count()
print(f"ref_nonre records: {row_count_nonre:,}  (run_date={run_date})")
assert row_count_nonre > 0, f"0 rows — check run_date={run_date} and active status filters"
display(df_nonre.limit(5))

In [0]:
from pyspark.sql import functions as F, Window

TARGET_RES = f"`{env}-cdh-delivery`.`it`.`l4738_ref_res`"

w_reb_res = Window.partitionBy("RealEstateKey").orderBy(
    F.when((F.col("RealEstateBuildingType") == "Real Estate") & F.col("SourceTableName").isNull(), 0).otherwise(1),
    "RealEstateBuildingKey"
)

reb_res = (
    spark.table(f"`{env}-cdh`.primaryactive.realestatebuilding")
    .filter((F.col("BusinessDate") == run_date) & F.col("RealEstateActive").isin("true", "Active"))
    .withColumn("rn", F.row_number().over(w_reb_res))
    .filter(F.col("rn") == 1).drop("rn")
    .select(
        F.col("RealEstateKey").cast("string").alias("reb_key"),
        F.col("RealEstateCepalKey").alias("PropertyKey"),
        F.col("RealEstateCategory").alias("ParentCategory"),
        F.col("RealEstateSubCategory").alias("PropertyCategory"),
        F.col("RealEstateDescription").alias("PropertyDescription"),
        F.col("RealEstateMarketValue").cast("decimal(19,2)").alias("MarketValue"),
        F.coalesce(F.col("RealEstateValueDate").cast("date"), F.col("RealEstateEvaluationDate").cast("date"), F.lit("1900-01-01").cast("date")).alias("LastValuationDate"),
        F.col("RealEstateCity").alias("city"),
        F.col("RealEstateArea").alias("area"),
        F.col("RealEstatePostalCode").alias("ZipCode"),
        F.col("RealEstateAddress1").alias("Address"),
        F.coalesce(F.col("RealEstateDistrict"), F.lit("N/A")).alias("Prefecture"),
    )
)

rei = (
    spark.table(f"`{env}-cdh`.primaryactive.realestateinfo")
    .filter(
        (F.col("BusinessDate") == run_date)
        & F.col("RealEstateKey").isNotNull()
        & F.col("OwnerPartyKey").isNotNull()
        & F.col("SourceSystem").isin("thesis", "thesis_RES")
    )
    .select(
        F.col("RealEstateKey").cast("string").alias("rei_key"),
        F.col("OwnerPartyKey").cast("string").alias("rei_OwnerPartyKey"),
        F.coalesce(F.col("OwnershipKey"), F.col("OwnershipReferenceKey")).cast("int").alias("PropertyOwnershipId"),
        F.col("OwnershipPercentage").cast("decimal(19,6)").alias("OwnershipPercentage"),
        F.col("OwnershipType").alias("OwnershipType"),
    )
)

asset_pkeys = df_asset.select(F.col("PropertyKey").alias("excl_key")).dropDuplicates()

df_res = (
    rei
    .join(reb_res, F.col("rei_key") == F.col("reb_key"), "inner")
    .join(pty, F.col("rei_OwnerPartyKey") == pty["pty_key"], "inner")
    .join(pf_lookup, pty["pty_PortfolioKey"].cast("int") == F.col("pf_thesis_id"), "inner")
    .join(asset_pkeys, F.col("PropertyKey") == F.col("excl_key"), "left_anti")
    .filter(pty["OwnerTaxNumber"].isNotNull())
    .select(
        F.lit(run_date).cast("date").alias("datadate"),
        F.col("pf_PortfolioId").cast("int").alias("PortfolioId"),
        F.col("pf_TiresiasName").alias("Portfolio"),
        F.col("pf_vendorCode").alias("vendorCode"),
        F.col("PropertyKey"),
        F.col("ParentCategory"), F.col("PropertyCategory"), F.col("PropertyDescription"),
        F.col("MarketValue"), F.col("LastValuationDate"),
        F.col("city"), F.col("area"), F.col("ZipCode"), F.col("Address"), F.col("Prefecture"),
        F.col("PropertyOwnershipId"),
        pty["OwnerTaxNumber"], pty["PropertyOwnerName"],
        F.col("OwnershipPercentage"), F.col("OwnershipType"),
        F.current_date().alias("EtlLoadingDate"),
    )
    .dropDuplicates()
)

row_count_res = df_res.count()
print(f"ref_res records: {row_count_res:,}  (run_date={run_date})")
assert row_count_res > 0, f"0 rows — check run_date={run_date}"
display(df_res.limit(5))

In [0]:
# ── ref_asset ──────────────────────────────────────────────────────────────────
(df_asset.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_ASSET))
print(f"\u2705 Written {row_count:,} rows \u2192 {TARGET_ASSET}  (datadate={run_date})")

# ── ref_nonre ──────────────────────────────────────────────────────────────────
(df_nonre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_NONRE))
print(f"\u2705 Written {row_count_nonre:,} rows \u2192 {TARGET_NONRE}  (datadate={run_date})")

# ── ref_res ──────────────────────────────────────────────────────────────────
(df_res.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_RES))
print(f"\u2705 Written {row_count_res:,} rows \u2192 {TARGET_RES}  (datadate={run_date})")